# Analise de negocio

Este notebook responde as duas hipoteses de negocio definidas para este MVP, substituindo as
perguntas genericas da versao anterior (atraso->satisfacao simples; vendas por estado):

1. **Frete vs. porte fisico do produto:** quais categorias pagam mais frete por quilo/volume
   transportado, e isso e proporcional ao porte fisico ou indica ineficiencia logistica/pricing?
2. **Concentracao de problemas por vendedor:** um grupo pequeno de vendedores concentra a maior
   parte dos atrasos e notas baixas (padrao 80/20), ou o problema esta distribuido por todos?

Fontes: `${catalog}.olist_gold.fato_vendas` e `${catalog}.olist_gold.dim_produtos`. Cada consulta
e seguida de uma celula markdown de interpretacao, a ser preenchida com os valores reais obtidos
na execucao no Databricks.


In [0]:
-- Pergunta 1: custo de frete relativo ao porte fisico, por categoria de produto.
WITH item_metrics AS (
  SELECT
    f.order_id,
    f.product_id,
    f.freight_value,
    p.product_category_name,
    p.product_weight_g,
    (p.product_length_cm * p.product_height_cm * p.product_width_cm) / 1000000.0 AS volume_m3
  FROM ${catalog}.olist_gold.fato_vendas f
  JOIN ${catalog}.olist_gold.dim_produtos p USING (product_id)
  WHERE p.product_weight_g IS NOT NULL
    AND p.product_length_cm IS NOT NULL
    AND p.product_height_cm IS NOT NULL
    AND p.product_width_cm IS NOT NULL
),
categoria_agg AS (
  SELECT
    product_category_name,
    COUNT(*) AS total_itens,
    ROUND(AVG(freight_value), 2) AS frete_medio,
    ROUND(AVG(product_weight_g), 0) AS peso_medio_g,
    ROUND(AVG(volume_m3), 4) AS volume_medio_m3,
    ROUND(AVG(freight_value) / NULLIF(AVG(product_weight_g) / 1000.0, 0), 2) AS frete_medio_por_kg
  FROM item_metrics
  GROUP BY product_category_name
  HAVING COUNT(*) >= 30
)
SELECT * FROM categoria_agg ORDER BY frete_medio_por_kg DESC;


product_category_name,total_itens,frete_medio,peso_medio_g,volume_medio_m3,frete_medio_por_kg
telefonia,4545,15.67,262.0,0.0018,59.89
fashion_esporte,30,19.27,340.0,0.0078,56.68
fashion_underwear_e_moda_praia,131,14.63,276.0,0.0039,52.97
tablets_impressao_imagem,83,14.77,307.0,0.0059,48.11
dvds_blu_ray,64,20.14,488.0,0.0018,41.29
fashion_bolsas_e_acessorios,2031,15.48,381.0,0.0039,40.61
consoles_games,1137,17.44,450.0,0.0056,38.73
perfumaria,3419,15.86,480.0,0.0049,33.03
telefonia_fixa,264,17.57,565.0,0.0042,31.07
relogios_presentes,5991,16.78,581.0,0.0029,28.88


### Conclusao da Pergunta 1
As 3 categorias com **pior** `frete_medio_por_kg` sao `telefonia` (R$ 59,89/kg; peso medio 262 g,
volume medio 0,0018 m³), `fashion_esporte` (R$ 56,68/kg; 340 g) e
`fashion_underwear_e_moda_praia` (R$ 52,97/kg; 276 g) — todas categorias de itens leves e de
volume pequeno. As 3 **melhores** sao `moveis_escritorio` (R$ 3,56/kg; peso medio 11,4 kg),
`moveis_quarto` (R$ 4,25/kg; 10,0 kg) e `moveis_sala` (R$ 4,41/kg; 8,1 kg) — moveis pesados e
volumosos.

O padrao **nao e proporcional ao porte fisico**: entre essas pontas, o frete absoluto medio
cresce bem menos (de R$ 15,67 para R$ 40,55, ~2,6x) do que o peso (de 262 g para 11.390 g,
~43x). Isso indica que o custo de frete tem um piso/tarifa minima por envio que varia pouco com
peso ou volume, penalizando desproporcionalmente itens leves (telefonia, moda) e diluindo o
custo nos itens pesados (moveis). Portanto, mais do que baixa densidade fisica isolada, o
resultado aponta para uma **ineficiencia de precificacao do frete** nas categorias de itens
pequenos/leves — oportunidade de revisar a tarifa minima ou a estrutura de frete fixo para
pedidos leves.


In [0]:
-- Pergunta 2: concentracao de atrasos e notas baixas por vendedor.
WITH pedidos_vendedor AS (
  SELECT
    f.seller_id,
    f.order_id,
    f.delivery_status,
    MAX(CAST(f.review_score AS INT)) AS review_score
  FROM ${catalog}.olist_gold.fato_vendas f
  GROUP BY f.seller_id, f.order_id, f.delivery_status
),
seller_agg AS (
  SELECT
    seller_id,
    COUNT(*) AS total_pedidos,
    SUM(CASE WHEN delivery_status IN ('atraso_1_7_dias', 'atraso_mais_de_7_dias') THEN 1 ELSE 0 END) AS pedidos_atrasados,
    SUM(CASE WHEN review_score IN (1, 2) THEN 1 ELSE 0 END) AS pedidos_nota_baixa
  FROM pedidos_vendedor
  GROUP BY seller_id
  HAVING COUNT(*) >= 10
),
ranked AS (
  SELECT *, NTILE(10) OVER (ORDER BY pedidos_atrasados DESC, pedidos_nota_baixa DESC) AS decil_risco
  FROM seller_agg
)
SELECT
  decil_risco,
  COUNT(*) AS qtd_vendedores,
  SUM(total_pedidos) AS total_pedidos,
  SUM(pedidos_atrasados) AS total_atrasados,
  SUM(pedidos_nota_baixa) AS total_nota_baixa,
  ROUND(100.0 * SUM(pedidos_atrasados) / SUM(SUM(pedidos_atrasados)) OVER (), 2) AS pct_do_total_atrasos,
  ROUND(100.0 * SUM(pedidos_nota_baixa) / SUM(SUM(pedidos_nota_baixa)) OVER (), 2) AS pct_do_total_notas_baixas
FROM ranked
GROUP BY decil_risco
ORDER BY decil_risco;


decil_risco,qtd_vendedores,total_pedidos,total_atrasados,total_nota_baixa,pct_do_total_atrasos,pct_do_total_notas_baixas
1,128,46571,3640,7232,58.97,53.62
2,127,13304,986,2001,15.97,14.84
3,127,8325,564,1156,9.14,8.57
4,127,6553,361,936,5.85,6.94
5,127,4367,254,559,4.11,4.14
6,127,4537,165,619,2.67,4.59
7,127,2786,127,301,2.06,2.23
8,127,3092,76,353,1.23,2.62
9,127,2602,0,273,0.00,2.02
10,127,1783,0,58,0.00,0.43


### Conclusao da Pergunta 2
O `decil_risco = 1` (128 vendedores, ~10% dos 1.271 vendedores com pelo menos 10 pedidos)
concentra **58,97%** de todos os pedidos atrasados e **53,62%** de todas as notas baixas (1-2)
da base — bem acima dos 10% esperados caso o problema estivesse distribuido uniformemente.
Somando os decis 1+2 (20% dos vendedores), a concentracao sobe para ~74,9% dos atrasos e ~68,5%
das notas baixas. Nos decis 9 e 10 (as 20% melhores fatias de vendedores) nao ha nenhum pedido
atrasado.

Isso **confirma o padrao 80/20**: o problema de atraso e insatisfacao esta fortemente
concentrado em um grupo pequeno de vendedores, e nao distribuido pela base. Recomendacao:
priorizar auditoria e plano de acao comercial/logistico com os ~128 vendedores do decil 1
(renegociacao de SLA, revisao da transportadora usada por eles, ou desligamento em casos
extremos), em vez de uma revisao ampla e generica de todos os vendedores.


## Resposta executiva

- **Frete vs. porte fisico:** o custo de frete **nao e proporcional ao porte fisico** do produto.
  Categorias leves e de pouco volume (`telefonia`, `fashion_esporte`,
  `fashion_underwear_e_moda_praia`) pagam entre R$ 53 e R$ 60 por kg, enquanto categorias de
  moveis pesados (`moveis_escritorio`, `moveis_quarto`, `moveis_sala`) pagam entre R$ 3,56 e
  R$ 4,41 por kg — ate 17x menos. O frete absoluto cresce muito pouco entre essas pontas (~2,6x)
  frente ao peso (~43x), o que aponta para uma tarifa minima/fixa por envio que penaliza itens
  leves. Recomendacao: revisar a precificacao de frete para categorias leves/pequenas (tarifa
  minima, frete fixo), em vez de assumir que o custo atual reflete corretamente o porte fisico.

- **Concentracao por vendedor:** confirma-se o **padrao 80/20**. O decil de maior risco (128
  vendedores, ~10% dos 1.271 vendedores com pelo menos 10 pedidos) concentra 58,97% de todos os
  pedidos atrasados e 53,62% de todas as notas baixas (1-2) da base; somando os dois piores
  decis (20% dos vendedores) a concentracao chega a ~74,9% dos atrasos e ~68,5% das notas
  baixas, enquanto os dois melhores decis nao registram nenhum atraso. Recomendacao: priorizar
  auditoria e plano de acao comercial/logistico focado nesses ~128 vendedores do decil 1
  (renegociacao de SLA, revisao de transportadora, ou desligamento em casos extremos), em vez de
  uma revisao ampla e generica de toda a base de vendedores.
